# Approximation algorithms for the knapsack problem (C++)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/combinatorial-optimization/blob/main/notebooks/knapsack-approximation_cpp.ipynb)

Executable notebook of the chapter “Approximation algorithms for the knapsack problem” (part “Approximation algorithms” of the course *Combinatorial Optimization*): it contains the approximation algorithms and schemes of the chapter for the knapsack problem: the greedy algorithm, Ext-Greedy, $H^2$, $H^{\ell}$, dynamic programming by profits and Basic-FPTAS, written in C++ and run on every example of the chapter and on every instance of the exercises, checking each result against the expected one.

The full chapter — pseudo-code, examples, proofs — is [on the website](https://fabiofurini.github.io/combinatorial-optimization/approximation-algorithms/knapsack-approximation/). The same notebook is also available in [Python](https://colab.research.google.com/github/fabiofurini/combinatorial-optimization/blob/main/notebooks/knapsack-approximation_python.ipynb).

**How to run it.** On Colab click the badge and then *Runtime → Run all*, or run the cells one at a time with `Shift+Enter`. Each C++ file is written to the folder `src/` by its `%%writefile` cell; the cell after `Main.cpp` compiles everything with `g++ -std=c++17 -O2` and runs the examples. Nothing to install (Colab already has `g++`).

**Indices.** In the notes items, jobs, cities, machines and bins are numbered from 1, here from 0: the algorithmic logic is unchanged.

## Declarations

The header declares all the functions of the chapter; each function is in its own file, with the same name.

In [1]:
%%writefile src/KP_heuristics.h
#ifndef KP_HEURISTICS_HEADER
#define KP_HEURISTICS_HEADER

// approximation algorithms and schemes for the knapsack problem
// items are indexed 0, 1, ..., n-1 and are assumed to be sorted by non-increasing efficiency
// every function returns the profit of the solution found and stores it in x
// (x[i] = 1 if item i is selected, 0 otherwise)

int GreedyKP(int n, int c, int* w, int* p, int* x);
int ExtendedGreedyKP(int n, int c, int* w, int* p, int* x);
int H2KP(int n, int c, int* w, int* p, int* x);
int HlKP(int n, int c, int* w, int* p, int l, int* x);
int DPProfits(int n, int c, int* w, int* p, int qmax, int* x);
int BasicFPTAS(int n, int c, int* w, int* p, int eps_num, int eps_den, int* x);

#endif

Writing src/KP_heuristics.h


## The greedy algorithm

The items are assumed to be sorted by non-increasing efficiency.

In [2]:
%%writefile src/GreedyKP.cpp
#include "KP_heuristics.h"

// items are assumed to be sorted by non-increasing efficiency
// x[i] stores the selection decision for item i
// the function returns the profit of the solution x

int GreedyKP(int n, int c, int* w, int* p, int* x)
{
    for (int i = 0; i < n; ++i)
    {
        x[i] = 0;                       // item i is initially not selected
    }
    int rc = c;                         // residual capacity

    for (int i = 0; i < n; ++i)
    {
        if (w[i] <= rc)
        {
            x[i] = 1;                   // select item i
            rc = rc - w[i];             // update residual capacity
        }
    }

    // profit of the solution x, returned for convenience
    int z = 0;
    for (int i = 0; i < n; ++i)
    {
        if (x[i] == 1) z = z + p[i];
    }
    return z;
}

Writing src/GreedyKP.cpp


## The Ext-Greedy algorithm

The better of the greedy solution and of the most profitable item alone.

In [3]:
%%writefile src/ExtendedGreedyKP.cpp
#include "KP_heuristics.h"

// items are assumed to be sorted by non-increasing efficiency
// x[i] stores the selection decision for item i
// the function returns the profit of the solution x

int ExtendedGreedyKP(int n, int c, int* w, int* p, int* x)
{
    GreedyKP(n, c, w, p, x);            // greedy solution

    int z = 0;                          // value of the greedy solution
    for (int i = 0; i < n; ++i)
    {
        if (x[i] == 1)
        {
            z = z + p[i];               // add the profit of a selected item
        }
    }

    int im = 0;                         // best single item
    int pm = p[0];
    for (int i = 1; i < n; ++i)
    {
        if (p[i] > pm)
        {
            im = i;                     // update best single item
            pm = p[i];                  // update its profit
        }
    }

    int* xhat = new int[n];             // best single-item solution
    for (int i = 0; i < n; ++i)
    {
        xhat[i] = 0;
    }
    xhat[im] = 1;                       // select only item im

    if (z < pm)
    {
        for (int i = 0; i < n; ++i)
        {
            x[i] = xhat[i];             // copy component by component
        }
        z = pm;
    }

    delete[] xhat;
    return z;
}

Writing src/ExtendedGreedyKP.cpp


## The $H^2$ algorithm

It calls Ext-Greedy on the residual instance of every feasible pair, exactly as the pseudo-code does, and it keeps the best solution found.

In [4]:
%%writefile src/H2KP.cpp
#include "KP_heuristics.h"

// items are assumed to be sorted by non-increasing efficiency
// the function returns H2(I), the value of the best solution found;
// in addition, x stores that solution, which the pseudo-code does not record

int H2KP(int n, int c, int* w, int* p, int* x)
{
    int im = 0;
    for (int j = 1; j < n; ++j)
    {
        if (p[j] > p[im]) im = j;
    }
    int pmax = p[im];
    int lb = pmax;                                  // best value found
    for (int j = 0; j < n; ++j) x[j] = 0;           // record the solution
    x[im] = 1;

    // residual instance I_ik: items, profits, weights and its solution
    int* idx = new int[n];
    int* pr  = new int[n];
    int* wr  = new int[n];
    int* xr  = new int[n];

    for (int i = 0; i < n; ++i)
    {
        for (int k = i + 1; k < n; ++k)
        {
            if (w[i] + w[k] > c) continue;          // the pair (i,k) is not in P

            int cbar = c - w[i] - w[k];             // residual capacity
            int pmin = (p[i] < p[k] ? p[i] : p[k]);

            // residual set I(i,k)
            int m = 0;
            for (int j = 0; j < n; ++j)
            {
                if (j != i && j != k && p[j] <= pmin && w[j] <= cbar)
                {
                    idx[m] = j; pr[m] = p[j]; wr[m] = w[j]; m++;
                }
            }

            // value of GR^e on the residual instance
            int zres = (m > 0) ? ExtendedGreedyKP(m, cbar, wr, pr, xr) : 0;
            int lbik = p[i] + p[k] + zres;

            if (lbik > lb)
            {
                lb = lbik;
                for (int j = 0; j < n; ++j) x[j] = 0;   // record the solution
                x[i] = 1; x[k] = 1;
                for (int j = 0; j < m; ++j) if (xr[j] == 1) x[idx[j]] = 1;
            }
        }
    }

    delete[] idx; delete[] pr; delete[] wr; delete[] xr;
    return lb;
}

Writing src/H2KP.cpp


## The $H^{\ell}$ algorithm

`NextSubset` enumerates the subsets of exactly $r$ items in lexicographic order; the preprocessing examines the subsets of at most $\ell-1$ items, and each subset of exactly $\ell$ items is completed by Ext-Greedy on its residual instance, as in the pseudo-code.

In [5]:
%%writefile src/HlKP.cpp
#include "KP_heuristics.h"

// items are assumed to be sorted by non-increasing efficiency
// l is the number of guessed items, with 2 <= l <= n
// the function returns H^l(I), the value of the best solution found;
// in addition, x stores that solution, which the pseudo-code does not record

// next subset of r indices of {0,...,n-1} in lexicographic order (false at the end)
static bool NextSubset(int* L, int r, int n)
{
    int t = r - 1;
    while (t >= 0 && L[t] == n - r + t) t--;
    if (t < 0) return false;
    L[t]++;
    for (int u = t + 1; u < r; ++u) L[u] = L[u - 1] + 1;
    return true;
}

int HlKP(int n, int c, int* w, int* p, int l, int* x)
{
    int* L    = new int[l];
    char* inL = new char[n];
    int* idx  = new int[n];
    int* pr   = new int[n];
    int* wr   = new int[n];
    int* xr   = new int[n];

    // lb := best value of the feasible sets of at most l-1 items
    int lb = 0;
    for (int j = 0; j < n; ++j) x[j] = 0;
    for (int r = 1; r <= l - 1; ++r)
    {
        for (int t = 0; t < r; ++t) L[t] = t;
        do
        {
            int wS = 0, pS = 0;
            for (int t = 0; t < r; ++t) { wS += w[L[t]]; pS += p[L[t]]; }
            if (wS <= c && pS > lb)
            {
                lb = pS;
                for (int j = 0; j < n; ++j) x[j] = 0;   // record the solution
                for (int t = 0; t < r; ++t) x[L[t]] = 1;
            }
        } while (NextSubset(L, r, n));
    }

    // foreach L in T(l): the feasible sets of exactly l items
    for (int t = 0; t < l; ++t) L[t] = t;
    do
    {
        int wL = 0, pL = 0, pmin = p[L[0]];
        for (int j = 0; j < n; ++j) inL[j] = 0;
        for (int t = 0; t < l; ++t)
        {
            wL += w[L[t]]; pL += p[L[t]]; inL[L[t]] = 1;
            if (p[L[t]] < pmin) pmin = p[L[t]];
        }
        if (wL > c) continue;                       // L is not in T(l)

        int cbar = c - wL;                          // residual capacity

        // residual set I(L)
        int m = 0;
        for (int j = 0; j < n; ++j)
        {
            if (!inL[j] && p[j] <= pmin && w[j] <= cbar)
            {
                idx[m] = j; pr[m] = p[j]; wr[m] = w[j]; m++;
            }
        }

        // complete the guess
        int lbL = pL + ((m > 0) ? ExtendedGreedyKP(m, cbar, wr, pr, xr) : 0);

        if (lbL > lb)
        {
            lb = lbL;
            for (int j = 0; j < n; ++j) x[j] = inL[j];  // record the solution
            for (int j = 0; j < m; ++j) if (xr[j] == 1) x[idx[j]] = 1;
        }
    } while (NextSubset(L, l, n));

    delete[] L; delete[] inL; delete[] idx; delete[] pr; delete[] wr; delete[] xr;
    return lb;
}

Writing src/HlKP.cpp


## Dynamic programming by profits

The second DP algorithm of the dynamic programming chapters, used by Basic-FPTAS. The table `u[i][q]` stores $u(i,q)$, and an optimal solution is reconstructed backwards. The parameter `qmax` truncates the table: only the profits $q\le\min\{P, qmax\}$ are computed, where $P$ is the total profit.

In [6]:
%%writefile src/DPProfits.cpp
#include "KP_heuristics.h"

// dynamic programming by profits (second DP algorithm of the dynamic programming notes)
// u(i,q) = minimum total weight of a subset of {0,...,i-1} with total profit exactly q
// the table is truncated at the profit qmax: only the values q = 0,...,min{P, qmax} are computed,
// where P is the total profit; qmax must be at least the optimal value of the instance
// the function returns the optimal value, i.e., the largest q with u(n,q) <= c,
// and stores an optimal solution in x

int DPProfits(int n, int c, int* w, int* p, int qmax, int* x)
{
    const int INF = 2147483647;
    int P = 0;                                  // total profit
    for (int i = 0; i < n; ++i) P += p[i];
    if (P > qmax) P = qmax;                     // truncation of the table

    // table u[i][q], i = 0,...,n and q = 0,...,P (P = min{total profit, qmax})
    int** u = new int*[n + 1];
    for (int i = 0; i <= n; ++i) u[i] = new int[P + 1];

    u[0][0] = 0;
    for (int q = 1; q <= P; ++q) u[0][q] = INF; // no item: only q = 0 is attained

    for (int i = 1; i <= n; ++i)
    {
        for (int q = 0; q <= P; ++q)
        {
            u[i][q] = u[i - 1][q];              // item i-1 is not selected
            if (p[i - 1] <= q && u[i - 1][q - p[i - 1]] != INF
                && w[i - 1] + u[i - 1][q - p[i - 1]] < u[i][q])
            {
                u[i][q] = w[i - 1] + u[i - 1][q - p[i - 1]];   // item i-1 is selected
            }
        }
    }

    // optimal value: the largest profit attained within the capacity
    int qstar = 0;
    for (int q = 0; q <= P; ++q) if (u[n][q] <= c) qstar = q;

    // reconstruct an optimal solution backwards
    int q = qstar;
    for (int i = n; i >= 1; --i)
    {
        if (u[i][q] == u[i - 1][q]) x[i - 1] = 0;
        else { x[i - 1] = 1; q -= p[i - 1]; }
    }

    for (int i = 0; i <= n; ++i) delete[] u[i];
    delete[] u;
    return qstar;
}

Writing src/DPProfits.cpp


## The Basic-FPTAS algorithm

The accuracy $\varepsilon$ is passed as a fraction $\varepsilon_{num}/\varepsilon_{den}$, so that the scaled profits $\lfloor p_j/K\rfloor$ are computed in exact integer arithmetic, and the table of `DPProfits` is truncated at $\lfloor 2n/\varepsilon\rfloor$.

In [7]:
%%writefile src/BasicFPTAS.cpp
#include "KP_heuristics.h"

// items are assumed to be sorted by non-increasing efficiency
// the accuracy is eps = eps_num / eps_den, with 0 < eps < 1
// x[i] stores the selection decision for item i
// the function returns the profit, in the original instance, of the solution x

int BasicFPTAS(int n, int c, int* w, int* p, int eps_num, int eps_den, int* x)
{
    int lb = ExtendedGreedyKP(n, c, w, p, x);   // value of the Ext-Greedy solution

    // scaling factor K = eps * lb / n and scaled profits p'[j] = floor(p[j] / K);
    // with eps = eps_num / eps_den, p[j] / K = p[j] * n * eps_den / (eps_num * lb),
    // which is computed in exact integer arithmetic
    int* ps = new int[n];                       // scaled profits p'
    for (int j = 0; j < n; ++j)
    {
        ps[j] = (int)(((long long)p[j] * n * eps_den) / ((long long)eps_num * lb));
    }

    // the optimal value of the scaled instance is at most floor(2n / eps): the table is truncated there
    int qmax = (int)((2LL * n * eps_den) / eps_num);

    DPProfits(n, c, w, ps, qmax, x);            // scaled instance solved to optimality

    // profit of the solution x in the original instance
    int z = 0;
    for (int j = 0; j < n; ++j)
    {
        if (x[j] == 1) z = z + p[j];
    }

    delete[] ps;
    return z;
}

Writing src/BasicFPTAS.cpp


## Execution on the examples of the chapter

Every example of the chapter and every instance of the exercises, with the expected result written in the code: each line shows the result found and `OK` when it is the expected one; the last line counts the checks and the errors.

In [8]:
%%writefile src/Main.cpp
#include <cstdio>
#include "KP_heuristics.h"

// runs every example of the lecture notes and every instance of the exercises,
// and compares the value found with the expected one

static int errors = 0, checks = 0;

static void check(const char* instance, const char* algorithm, int found, int expected)
{
    checks++;
    if (found != expected) errors++;
    printf("%-22s %-14s %4d %s\n", instance, algorithm, found,
           found == expected ? "OK" : "<-- EXPECTED A DIFFERENT VALUE");
}

int main()
{
    int x[16];
    { int p[] = {6,5,8,9,6,7,3}; int w[] = {2,3,6,7,5,9,4};
      check("Example 1 (c=9)", "GR", GreedyKP(7, 9, w, p, x), 14);
      check("Example 1 (c=9)", "GRe", ExtendedGreedyKP(7, 9, w, p, x), 14);
      check("Example 1 (c=9)", "H2", H2KP(7, 9, w, p, x), 15);
      check("Example 1 (c=9)", "Hl^2", HlKP(7, 9, w, p, 2, x), 15);
      check("Example 1 (c=9)", "Hl^3", HlKP(7, 9, w, p, 3, x), 15);
    }
    { int p[] = {6,8,19,4}; int w[] = {2,3,9,5};
      check("Example 2", "GR", GreedyKP(4, 10, w, p, x), 18);
      check("Example 2", "GRe", ExtendedGreedyKP(4, 10, w, p, x), 19);
      check("Example 2", "H2", H2KP(4, 10, w, p, x), 19);
      check("Example 2", "Hl^2", HlKP(4, 10, w, p, 2, x), 19);
      check("Example 2", "Hl^3", HlKP(4, 10, w, p, 3, x), 19);
    }
    { int p[] = {4,6,5,12,3}; int w[] = {1,2,2,6,2};
      check("Example 3", "GR", GreedyKP(5, 10, w, p, x), 18);
      check("Example 3", "GRe", ExtendedGreedyKP(5, 10, w, p, x), 18);
      check("Example 3", "H2", H2KP(5, 10, w, p, x), 23);
      check("Example 3", "Hl^2", HlKP(5, 10, w, p, 2, x), 23);
      check("Example 3", "Hl^3", HlKP(5, 10, w, p, 3, x), 23);
    }
    { int p[] = {4,6,4,3,2,20}; int w[] = {1,2,2,2,2,21};
      check("Example 4", "GR", GreedyKP(6, 21, w, p, x), 19);
      check("Example 4", "GRe", ExtendedGreedyKP(6, 21, w, p, x), 20);
      check("Example 4", "H2", H2KP(6, 21, w, p, x), 20);
      check("Example 4", "Hl^2", HlKP(6, 21, w, p, 2, x), 20);
      check("Example 4", "Hl^3", HlKP(6, 21, w, p, 3, x), 20);
    }
    { int p[] = {6,5,8,9,6,7,3}; int w[] = {2,3,6,7,5,9,4};
      check("Example H2 (c=12)", "GR", GreedyKP(7, 12, w, p, x), 19);
      check("Example H2 (c=12)", "GRe", ExtendedGreedyKP(7, 12, w, p, x), 19);
      check("Example H2 (c=12)", "H2", H2KP(7, 12, w, p, x), 20);
      check("Example H2 (c=12)", "Hl^2", HlKP(7, 12, w, p, 2, x), 20);
      check("Example H2 (c=12)", "Hl^3", HlKP(7, 12, w, p, 3, x), 20);
    }
    { int p[] = {2,3,3,3,3}; int w[] = {1,3,3,3,3};
      check("Example H^l", "GR", GreedyKP(5, 12, w, p, x), 11);
      check("Example H^l", "GRe", ExtendedGreedyKP(5, 12, w, p, x), 11);
      check("Example H^l", "H2", H2KP(5, 12, w, p, x), 11);
      check("Example H^l", "Hl^2", HlKP(5, 12, w, p, 2, x), 11);
      check("Example H^l", "Hl^3", HlKP(5, 12, w, p, 3, x), 12);
    }
    { int p[] = {19,19,14,14}; int w[] = {2,4,3,3};
      check("Example FPTAS", "GR", GreedyKP(4, 8, w, p, x), 38);
      check("Example FPTAS", "GRe", ExtendedGreedyKP(4, 8, w, p, x), 38);
      check("Example FPTAS", "H2", H2KP(4, 8, w, p, x), 47);
      check("Example FPTAS", "Hl^2", HlKP(4, 8, w, p, 2, x), 47);
      check("Example FPTAS", "Hl^3", HlKP(4, 8, w, p, 3, x), 47);
      check("Example FPTAS", "FP(eps=1/2)", BasicFPTAS(4, 8, w, p, 1, 2, x), 38);
    }
    { int p[] = {12,6,6,8,6}; int w[] = {3,2,3,9,7};
      check("EX_GR_1", "GR", GreedyKP(5, 15, w, p, x), 30);
      check("EX_GR_1", "GRe", ExtendedGreedyKP(5, 15, w, p, x), 30);
      check("EX_GR_1", "H2", H2KP(5, 15, w, p, x), 30);
      check("EX_GR_1", "Hl^2", HlKP(5, 15, w, p, 2, x), 30);
      check("EX_GR_1", "Hl^3", HlKP(5, 15, w, p, 3, x), 30);
    }
    { int p[] = {10,6,3,7,11,2}; int w[] = {1,1,1,4,8,3};
      check("EX_GR_2", "GR", GreedyKP(6, 13, w, p, x), 28);
      check("EX_GR_2", "GRe", ExtendedGreedyKP(6, 13, w, p, x), 28);
      check("EX_GR_2", "H2", H2KP(6, 13, w, p, x), 30);
      check("EX_GR_2", "Hl^2", HlKP(6, 13, w, p, 2, x), 30);
      check("EX_GR_2", "Hl^3", HlKP(6, 13, w, p, 3, x), 30);
    }
    { int p[] = {8,7,18,14,10}; int w[] = {3,3,9,8,7};
      check("EX_GR_3", "GR", GreedyKP(5, 12, w, p, x), 15);
      check("EX_GR_3", "GRe", ExtendedGreedyKP(5, 12, w, p, x), 18);
      check("EX_GR_3", "H2", H2KP(5, 12, w, p, x), 26);
      check("EX_GR_3", "Hl^2", HlKP(5, 12, w, p, 2, x), 26);
      check("EX_GR_3", "Hl^3", HlKP(5, 12, w, p, 3, x), 26);
    }
    { int p[] = {11,5,3,4,3}; int w[] = {8,4,3,9,8};
      check("EX_GRe_1", "GR", GreedyKP(5, 22, w, p, x), 19);
      check("EX_GRe_1", "GRe", ExtendedGreedyKP(5, 22, w, p, x), 19);
      check("EX_GRe_1", "H2", H2KP(5, 22, w, p, x), 20);
      check("EX_GRe_1", "Hl^2", HlKP(5, 22, w, p, 2, x), 20);
      check("EX_GRe_1", "Hl^3", HlKP(5, 22, w, p, 3, x), 20);
    }
    { int p[] = {16,19,18,14,8}; int w[] = {5,8,8,7,6};
      check("EX_GRe_2", "GR", GreedyKP(5, 8, w, p, x), 16);
      check("EX_GRe_2", "GRe", ExtendedGreedyKP(5, 8, w, p, x), 19);
      check("EX_GRe_2", "H2", H2KP(5, 8, w, p, x), 19);
      check("EX_GRe_2", "Hl^2", HlKP(5, 8, w, p, 2, x), 19);
      check("EX_GRe_2", "Hl^3", HlKP(5, 8, w, p, 3, x), 19);
    }
    { int p[] = {2,8,8}; int w[] = {1,8,8};
      check("EX_GRe_3", "GR", GreedyKP(3, 16, w, p, x), 10);
      check("EX_GRe_3", "GRe", ExtendedGreedyKP(3, 16, w, p, x), 10);
      check("EX_GRe_3", "H2", H2KP(3, 16, w, p, x), 16);
      check("EX_GRe_3", "Hl^2", HlKP(3, 16, w, p, 2, x), 16);
      check("EX_GRe_3", "Hl^3", HlKP(3, 16, w, p, 3, x), 16);
    }
    { int p[] = {8,9,2,3,2}; int w[] = {5,8,3,7,9};
      check("EX_H2_1", "GR", GreedyKP(5, 11, w, p, x), 10);
      check("EX_H2_1", "GRe", ExtendedGreedyKP(5, 11, w, p, x), 10);
      check("EX_H2_1", "H2", H2KP(5, 11, w, p, x), 11);
      check("EX_H2_1", "Hl^2", HlKP(5, 11, w, p, 2, x), 11);
      check("EX_H2_1", "Hl^3", HlKP(5, 11, w, p, 3, x), 11);
    }
    { int p[] = {14,9,5,10,15}; int w[] = {1,1,2,5,8};
      check("EX_H2_2", "GR", GreedyKP(5, 15, w, p, x), 38);
      check("EX_H2_2", "GRe", ExtendedGreedyKP(5, 15, w, p, x), 38);
      check("EX_H2_2", "H2", H2KP(5, 15, w, p, x), 43);
      check("EX_H2_2", "Hl^2", HlKP(5, 15, w, p, 2, x), 43);
      check("EX_H2_2", "Hl^3", HlKP(5, 15, w, p, 3, x), 48);
    }
    { int p[] = {13,5,8,5,7}; int w[] = {1,1,3,2,5};
      check("EX_H2_3", "GR", GreedyKP(5, 10, w, p, x), 31);
      check("EX_H2_3", "GRe", ExtendedGreedyKP(5, 10, w, p, x), 31);
      check("EX_H2_3", "H2", H2KP(5, 10, w, p, x), 31);
      check("EX_H2_3", "Hl^2", HlKP(5, 10, w, p, 2, x), 31);
      check("EX_H2_3", "Hl^3", HlKP(5, 10, w, p, 3, x), 33);
    }
    { int p[] = {14,13,13,10,11}; int w[] = {1,2,3,3,9};
      check("EX_Hl_1", "GR", GreedyKP(5, 16, w, p, x), 50);
      check("EX_Hl_1", "GRe", ExtendedGreedyKP(5, 16, w, p, x), 50);
      check("EX_Hl_1", "H2", H2KP(5, 16, w, p, x), 50);
      check("EX_Hl_1", "Hl^2", HlKP(5, 16, w, p, 2, x), 50);
      check("EX_Hl_1", "Hl^3", HlKP(5, 16, w, p, 3, x), 51);
    }
    { int p[] = {2,4,4,4,4,4}; int w[] = {1,4,4,4,4,4};
      check("EX_Hl_2", "GR", GreedyKP(6, 20, w, p, x), 18);
      check("EX_Hl_2", "GRe", ExtendedGreedyKP(6, 20, w, p, x), 18);
      check("EX_Hl_2", "H2", H2KP(6, 20, w, p, x), 18);
      check("EX_Hl_2", "Hl^2", HlKP(6, 20, w, p, 2, x), 18);
      check("EX_Hl_2", "Hl^3", HlKP(6, 20, w, p, 3, x), 18);
    }
    { int p[] = {15,15,14,8,12,13}; int w[] = {1,2,3,2,6,9};
      check("EX_Hl_3", "GR", GreedyKP(6, 18, w, p, x), 64);
      check("EX_Hl_3", "GRe", ExtendedGreedyKP(6, 18, w, p, x), 64);
      check("EX_Hl_3", "H2", H2KP(6, 18, w, p, x), 64);
      check("EX_Hl_3", "Hl^2", HlKP(6, 18, w, p, 2, x), 64);
      check("EX_Hl_3", "Hl^3", HlKP(6, 18, w, p, 3, x), 64);
    }
    { int p[] = {24,11,23,5}; int w[] = {2,3,9,3};
      check("EX_FP_1", "GR", GreedyKP(4, 13, w, p, x), 40);
      check("EX_FP_1", "GRe", ExtendedGreedyKP(4, 13, w, p, x), 40);
      check("EX_FP_1", "H2", H2KP(4, 13, w, p, x), 47);
      check("EX_FP_1", "Hl^2", HlKP(4, 13, w, p, 2, x), 47);
      check("EX_FP_1", "Hl^3", HlKP(4, 13, w, p, 3, x), 47);
      check("EX_FP_1", "FP(eps=1/2)", BasicFPTAS(4, 13, w, p, 1, 2, x), 47);
      check("EX_FP_1", "FP(eps=1/4)", BasicFPTAS(4, 13, w, p, 1, 4, x), 47);
    }
    { int p[] = {14,12,14,18}; int w[] = {3,4,5,7};
      check("EX_FP_2", "GR", GreedyKP(4, 16, w, p, x), 40);
      check("EX_FP_2", "GRe", ExtendedGreedyKP(4, 16, w, p, x), 40);
      check("EX_FP_2", "H2", H2KP(4, 16, w, p, x), 46);
      check("EX_FP_2", "Hl^2", HlKP(4, 16, w, p, 2, x), 46);
      check("EX_FP_2", "Hl^3", HlKP(4, 16, w, p, 3, x), 46);
      check("EX_FP_2", "FP(eps=1/2)", BasicFPTAS(4, 16, w, p, 1, 2, x), 44);
      check("EX_FP_2", "FP(eps=1/4)", BasicFPTAS(4, 16, w, p, 1, 4, x), 46);
    }
    { int p[] = {14,12,14,18}; int w[] = {3,4,5,7};
      check("EX_FP_3", "GR", GreedyKP(4, 16, w, p, x), 40);
      check("EX_FP_3", "GRe", ExtendedGreedyKP(4, 16, w, p, x), 40);
      check("EX_FP_3", "H2", H2KP(4, 16, w, p, x), 46);
      check("EX_FP_3", "Hl^2", HlKP(4, 16, w, p, 2, x), 46);
      check("EX_FP_3", "Hl^3", HlKP(4, 16, w, p, 3, x), 46);
      check("EX_FP_3", "FP(eps=1/2)", BasicFPTAS(4, 16, w, p, 1, 2, x), 44);
      check("EX_FP_3", "FP(eps=1/4)", BasicFPTAS(4, 16, w, p, 1, 4, x), 46);
    }

    printf("\n%d checks, %d errors\n", checks, errors);
    return errors == 0 ? 0 : 1;
}

Writing src/Main.cpp


In [9]:
!g++ -std=c++17 -O2 -o KP_HEUR src/*.cpp && ./KP_HEUR

Example 1 (c=9)        GR               14 OK
Example 1 (c=9)        GRe              14 OK
Example 1 (c=9)        H2               15 OK
Example 1 (c=9)        Hl^2             15 OK
Example 1 (c=9)        Hl^3             15 OK
Example 2              GR               18 OK
Example 2              GRe              19 OK
Example 2              H2               19 OK
Example 2              Hl^2             19 OK
Example 2              Hl^3             19 OK
Example 3              GR               18 OK
Example 3              GRe              18 OK
Example 3              H2               23 OK
Example 3              Hl^2             23 OK
Example 3              Hl^3             23 OK
Example 4              GR               19 OK
Example 4              GRe              20 OK
Example 4              H2               20 OK
Example 4              Hl^2             20 OK
Example 4              Hl^3             20 OK
Example H2 (c=12)      GR               19 OK
Example H2 (c=12)      GRe        

## Try it yourself

Change the data in the cell below and run it again.

In [10]:
%%writefile try.cpp
#include <cstdio>
#include "src/KP_heuristics.h"

// Try it yourself: change the data below, compile and run again
// (the items must be sorted by non-increasing efficiency p[i]/w[i], as the functions assume)
int c = 9;                          // knapsack capacity
int p[] = {6, 5, 8, 9, 6, 7, 3};    // profits
int w[] = {2, 3, 6, 7, 5, 9, 4};    // weights
int l = 3;                          // number of guessed items of H^l (2 <= l <= n)
int eps_num = 1, eps_den = 2;       // accuracy eps = eps_num / eps_den of the Basic-FPTAS
int n = sizeof(p) / sizeof(p[0]);

static void print_solution(const char* name, int z, const int* x)
{
    printf("%-15s z = %3d   x =", name, z);
    for (int i = 0; i < n; ++i) printf(" %d", x[i]);
    printf("\n");
}

int main()
{
    int* x = new int[n];
    char name[32];
    int P = 0;                      // total profit, used as qmax for the exact DP
    for (int i = 0; i < n; ++i) P += p[i];

    printf("Instance: c = %d, n = %d items\n", c, n);
    int z = GreedyKP(n, c, w, p, x);
    print_solution("GR", z, x);
    z = ExtendedGreedyKP(n, c, w, p, x);
    print_solution("GR^e", z, x);
    z = H2KP(n, c, w, p, x);
    print_solution("H^2", z, x);
    z = HlKP(n, c, w, p, l, x);
    snprintf(name, sizeof(name), "H^%d", l);
    print_solution(name, z, x);
    z = BasicFPTAS(n, c, w, p, eps_num, eps_den, x);
    snprintf(name, sizeof(name), "FPTAS eps=%d/%d", eps_num, eps_den);
    print_solution(name, z, x);
    z = DPProfits(n, c, w, p, P, x);
    print_solution("optimum (DP)", z, x);

    delete[] x;
    return 0;
}

Writing try.cpp


In [11]:
!g++ -std=c++17 -O2 -o try try.cpp $(ls src/*.cpp | grep -v Main.cpp) && ./try

Instance: c = 9, n = 7 items
GR              z =  14   x = 1 1 0 0 0 0 1
GR^e            z =  14   x = 1 1 0 0 0 0 1
H^2             z =  15   x = 1 0 0 1 0 0 0
H^3             z =  15   x = 1 0 0 1 0 0 0
FPTAS eps=1/2   z =  15   x = 1 0 0 1 0 0 0
optimum (DP)    z =  15   x = 1 0 0 1 0 0 0


---

Teaching material by [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza University of Rome.
Text, figures and data [CC BY 4.0](https://github.com/fabiofurini/combinatorial-optimization/blob/main/LICENSE), code [MIT](https://github.com/fabiofurini/combinatorial-optimization/blob/main/LICENSE-CODE).